# RSA Cryptosystem & Cryptanalytic Attacks
**Introduction to Number Theory and Cryptography Workshop**  
*Maths and Physics Club, IIT Bombay*

This notebook contains complete implementations and solutions for all 7 questions in the RSA Assignment:
1. **RSA Encryption** (Encrypt)
2. **RSA Decryption** (Decrypt)
3. **Small Message Space Attack** (DecipherSimple)
4. **Small Prime Factor Attack** (DecipherSmallPrime)
5. **Fermat Factorization Attack** for close primes (DecipherSmallDiff)
6. **Shared Prime / Common Divisor Attack** (DecipherCommonDivisor)
7. **H?stad's Broadcast Attack** via Chinese Remainder Theorem (DecipherHastad)

## Core Helper Functions
Conversion between strings and large integers, extended Euclidean algorithm, and modular inverse.

In [ ]:
import math

def ConvertToInt(message_str):
    res = 0
    for ch in message_str:
        res = res * 256 + ord(ch)
    return res

def ConvertToStr(n):
    res = ""
    while n > 0:
        res += chr(n % 256)
        n //= 256
    return res[::-1]

def PowMod(a, n, mod):
    return pow(a, n, mod)

def ExtendedEuclid(a, b):
    if b == 0:
        return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0:
        b = (b % n + n) % n
    return b

## Question 1: RSA Encryption

In [ ]:
def Encrypt(message, modulo, exponent):
    m = ConvertToInt(message)
    return PowMod(m, exponent, modulo)

# Verification
c_sample = Encrypt("attack", 101, 12)
print("Sample Ciphertext:", c_sample)

## Question 2: RSA Decryption

In [ ]:
def Decrypt(ciphertext, p, q, exponent):
    phi = (p - 1) * (q - 1)
    d = InvertModulo(exponent, phi)
    m = PowMod(ciphertext, d, p * q)
    return ConvertToStr(m)

p = 1000000007
q = 1000000009
exponent = 23917
modulo = p * q
ciphertext = Encrypt("attack", modulo, exponent)
message = Decrypt(ciphertext, p, q, exponent)
print("Decrypted message:", message)
assert message == "attack"

## Question 3: Small Message Space Brute-Force Attack

In [ ]:
def DecipherSimple(ciphertext, modulo, exponent, potential_messages):
    for msg in potential_messages:
        if Encrypt(msg, modulo, exponent) == ciphertext:
            return msg
    return "don't know"

modulo = 101
exponent = 12
ciphertext = Encrypt("attack", modulo, exponent)
res = DecipherSimple(ciphertext, modulo, exponent, ["attack", "don't attack", "wait"])
print("DecipherSimple recovered:", res)
assert res == "attack"

## Question 4: Small Prime Factor Attack ( < 10^6$)

In [ ]:
def DecipherSmallPrime(ciphertext, modulo, exponent):
    p = None
    for i in range(2, 1000001):
        if modulo % i == 0:
            p = i
            break
    if p is None:
        return "don't know"
    q = modulo // p
    return Decrypt(ciphertext, p, q, exponent)

small_p = 101
large_q = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
modulo = small_p * large_q
exponent = 239
ciphertext = Encrypt("attack", modulo, exponent)
recovered = DecipherSmallPrime(ciphertext, modulo, exponent)
print("DecipherSmallPrime recovered:", recovered)
assert recovered == "attack"

## Question 5: Fermat Factorization for Close Primes ($|p - q| < 5000$)

In [ ]:
def DecipherSmallDiff(ciphertext, modulo, exponent):
    a = math.isqrt(modulo)
    if a * a < modulo:
        a += 1
    while True:
        b2 = a * a - modulo
        b = math.isqrt(b2)
        if b * b == b2:
            p = a - b
            q = a + b
            break
        a += 1
    return Decrypt(ciphertext, p, q, exponent)

p = 1000000007
q = 1000000009
n = p * q
e = 239
ciphertext = Encrypt("attack", n, e)
message = DecipherSmallDiff(ciphertext, n, e)
print("DecipherSmallDiff recovered:", message)
assert message == "attack"

## Question 6: Shared Prime / Common Divisor Attack

In [ ]:
def GCD(a, b):
    while b:
        a, b = b, a % b
    return a

def DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent, second_ciphertext, second_modulo, second_exponent):
    p = GCD(first_modulo, second_modulo)
    assert p > 1, "Moduli do not share a prime factor"
    q1 = first_modulo // p
    q2 = second_modulo // p
    msg1 = Decrypt(first_ciphertext, p, q1, first_exponent)
    msg2 = Decrypt(second_ciphertext, p, q2, second_exponent)
    return (msg1, msg2)

p = 101
q1 = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
q2 = 1000000007
first_modulo = p * q1
second_modulo = p * q2
first_exponent = 239
second_exponent = 17
first_ciphertext = Encrypt("attack", first_modulo, first_exponent)
second_ciphertext = Encrypt("wait", second_modulo, second_exponent)
msg1, msg2 = DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent, second_ciphertext, second_modulo, second_exponent)
print("Recovered messages:", msg1, "|", msg2)
assert msg1 == "attack" and msg2 == "wait"

## Question 7: H?stad's Broadcast Attack for  = 2$

In [ ]:
def ChineseRemainderTheorem(n1, r1, n2, r2):
    (x, y) = ExtendedEuclid(n1, n2)
    r = (r1 * n2 * y + r2 * n1 * x) % (n1 * n2)
    return (r + n1 * n2) % (n1 * n2)

def DecipherHastad(first_ciphertext, first_modulo, second_ciphertext, second_modulo):
    C = ChineseRemainderTheorem(first_modulo, first_ciphertext, second_modulo, second_ciphertext)
    m = math.isqrt(C)
    return ConvertToStr(m)

p1 = 790383132652258876190399065097
q1 = 662503581792812531719955475509
p2 = 656917682542437675078478868539
q2 = 1263581691331332127259083713503
n1 = p1 * q1
n2 = p2 * q2
ciphertext1 = Encrypt("attack", n1, 2)
ciphertext2 = Encrypt("attack", n2, 2)
message = DecipherHastad(ciphertext1, n1, ciphertext2, n2)
print("DecipherHastad recovered:", message)
assert message == "attack"